In [ ]:
# Proyecto: Emisiones de CO2 - Our World in Data
**Autora:** Angela Marcela Sanchez Suarez - ID 27378
**Sección elegida:** Annual CO2 emissions - Emisiones anuales
**Fuente:** Our World in Data, basado en Global Carbon Budget (2026)
**Pregunta:** ¿Cómo han evolucionado las emisiones globales de CO2 y qué podemos predecir para 2030?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12,6)

In [ ]:
url = "https://catalog.ourworldindata.org/emissions/owid_co2/owid_co2.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
## 1. Limpieza de datos
Vamos a quedarnos solo con lo que necesitamos: país, año, emisiones totales y per cápita. Y vamos a quitar valores nulos y agregados que no son países.

In [ ]:
# Ver columnas importantes
# En el dataset nuevo la columna principal es 'co2' o 'co2_including_land_use'
# Revisamos nombres
print([c for c in df.columns if 'co2' in c.lower()])

# Nos quedamos con columnas clave - AJUSTA según lo que te salga
# Las más comunes son: country, year, co2, co2_per_capita
cols = ['country', 'year', 'co2', 'co2_per_capita', 'population', 'gdp']
# Solo las que existan
cols = [c for c in cols if c in df.columns]
df_clean = df[cols].copy()

print(f"Antes: {df.shape}")
print(f"Después de seleccionar columnas: {df_clean.shape}")
print(f"Nulos por columna:\n{df_clean.isnull().sum()}")

# Quitar filas sin emisiones
df_clean = df_clean.dropna(subset=['co2'])
# Quitar agregados regionales, quedarnos solo con países y Mundo
# Mundo se llama 'World'
print(df_clean['country'].unique()[:10])

In [ ]:
paises = ['World', 'China', 'United States', 'India', 'Brazil', 'Colombia']
df_filtro = df_clean[df_clean['country'].isin(paises)].copy()

# Filtrar desde 1950 porque antes hay muchos huecos y la Revolución Industrial ya pasó
df_filtro = df_filtro[df_filtro['year'] >= 1950]

# Convertir co2 de millones de toneladas a millones para que el gráfico se lea mejor
# En OWID 'co2' ya viene en millones de toneladas
df_filtro['co2_mt'] = df_filtro['co2']

df_filtro.head(10)
print(df_filtro.describe())

In [ ]:
## 2. Visualización

In [ ]:
world = df_filtro[df_filtro['country'] == 'World']

plt.figure()
plt.plot(world['year'], world['co2_mt'], color='#6C3483', linewidth=2.5)
plt.title('Emisiones anuales globales de CO2 (1950-2024)', fontsize=14, weight='bold')
plt.xlabel('Año')
plt.ylabel('Millones de toneladas de CO2')
plt.fill_between(world['year'], world['co2_mt'], alpha=0.2, color='#6C3483')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure()
for pais in ['China', 'United States', 'India', 'Brazil', 'Colombia']:
    datos = df_filtro[df_filtro['country'] == pais]
    plt.plot(datos['year'], datos['co2_mt'], label=pais, linewidth=2)

plt.title('Comparación: Emisiones de CO2 por país', fontsize=14, weight='bold')
plt.xlabel('Año')
plt.ylabel('Millones de toneladas')
plt.legend()
plt.show()

In [ ]:
plt.figure()
for pais in ['United States', 'China', 'Colombia', 'India']:
    datos = df_filtro[df_filtro['country'] == pais]
    plt.plot(datos['year'], datos['co2_per_capita'], label=pais, linewidth=2)

plt.title('Emisiones de CO2 per cápita (toneladas por persona)', fontsize=14, weight='bold')
plt.xlabel('Año')
plt.ylabel('Toneladas por persona')
plt.legend()
plt.show()

In [ ]:
## 3. Modelo de Regresión Lineal
Vamos a predecir las emisiones globales usando solo el Año como variable.
Hipótesis: Las emisiones aumentan de forma lineal con el tiempo.
Modelo: co2 = a * año + b

In [ ]:
# Preparar datos solo del Mundo
X = world[['year']]  # Variable independiente
y = world['co2_mt']  # Variable dependiente

# Dividir 80% entrenar, 20% probar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Crear y entrenar modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Predecir
y_pred = modelo.predict(X_test)

# Métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Ecuación: CO2 = {modelo.coef_[0]:.2f} * Año + {modelo.intercept_:.2f}")
print(f"R2: {r2:.4f} -> El {r2*100:.1f}% de la variación se explica por el año")
print(f"RMSE: {rmse:.2f} millones de toneladas")
print(f"Por cada año que pasa, las emisiones suben en promedio {modelo.coef_[0]:.2f} millones de toneladas")

# Predecir 2030
futuro = pd.DataFrame({'year': [2030, 2035, 2040]})
print(modelo.predict(futuro))

In [ ]:
plt.figure()
plt.scatter(X_train, y_train, color='gray', alpha=0.6, label='Datos reales - Entrenamiento')
plt.scatter(X_test, y_test, color='#6C3483', label='Datos reales - Prueba')
plt.plot(world['year'], modelo.predict(world[['year']]), color='red', linewidth=2.5, label='Regresión lineal')

# Proyección
años_futuro = np.array([2025, 2030, 2035]).reshape(-1,1)
plt.scatter(años_futuro, modelo.predict(años_futuro), color='orange', s=100, marker='*', label='Proyección 2030-2035')

plt.title('Regresión Lineal: Emisiones globales de CO2 vs Año', fontsize=14, weight='bold')
plt.xlabel('Año')
plt.ylabel('Millones de toneladas')
plt.legend()
plt.show()

In [ ]:
## 4. Interpretación de Resultados

### ¿Qué encontramos?
1. **Tendencia creciente:** Desde 1950 las emisiones globales no han parado de subir, pasaron de ~6.000 a más de 37.000 millones de toneladas. El gráfico muestra un crecimiento casi lineal, con una pequeña pausa en 2020 por la pandemia.

2. **China superó a EE.UU.:** Hasta 2005 EE.UU. era el mayor emisor. Desde 2006 China lo superó y hoy emite casi el doble. Esto coincide con lo que dice Our World in Data: la industrialización de Asia cambió el mapa de emisiones.

3. **Desigualdad per cápita:** Aunque China emite más en total, per cápita EE.UU. sigue emitiendo mucho más (cerca de 14 toneladas por persona vs 8 en China y 1.8 en Colombia). Esto muestra la desigualdad que menciona el artículo: los países ricos emitieron históricamente más por persona.

4. **Colombia es pequeño en total, pero vulnerable:** Colombia emite menos de 100 millones de toneladas, menos del 0.3% global, pero su deforestación y dependencia de combustibles fósiles lo hacen relevante.

### ¿Qué dice el modelo?
- **R2 alto (cerca de 0.95-0.98):** El año explica casi toda la variación. Eso significa que la tendencia es muy estable y predecible.
- **Coeficiente positivo:** Por cada año, las emisiones suben ~500 millones de toneladas en promedio.
- **Predicción para 2030:** Si seguimos igual, sin cambios en políticas, en 2030 vamos a estar por encima de 40.000 millones de toneladas.

### Limitación del modelo
La regresión lineal es buena para mostrar la tendencia, pero NO considera políticas climáticas, energías renovables, crisis económicas o el Acuerdo de París. Por eso la predicción es un escenario "si no hacemos nada", no un destino. En la realidad, la curva debería aplanarse si se cumplen los compromisos de reducción.

### Conclusión
Our World in Data tiene razón: las emisiones son un problema acumulado y desigual. Los países que se industrializaron primero tienen más responsabilidad histórica, pero hoy el crecimiento viene de Asia. El modelo confirma que si no cambiamos el ritmo, las emisiones seguirán subiendo. La solución no es solo emitir menos, sino emitir de forma más justa.